# Analysis-period surface energy budget

Computes reusable member-level surface-energy-budget caches and plots regional ensemble summaries. Increase `max_members` after a one-member smoke test.


## 1. Libraries


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt


## 2. User setup


In [ ]:
from pathlib import Path
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
LAND_SUPPORT_DIR = Path("/compyfs/zhan391/acme_init/lnd_sea_mask")   # land mask, soil-layer thickness
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")

workflow_name = "analysis_atm_model_evaluation"
run_segment = "da"
season = "S0"
experiments = ("CTRL10-S0", "DART10-S0", "DART20-S0", "DART40-S0")  # S1: ("DART40-S1",)
# Analysis runs are evaluated over the 64-cycle window (S0: 16-31 Dec 2011;
# S1: 16-31 May 2012), the same days as the DA scorecard. None = whole month.
use_analysis_window = True
max_members = None # all members, set to 1 for test 
plot_variables = ("Rn", "LE", "SH", "Residual", "BowenRatio")
region = "land"
force_compute = False
show = True

FIGURE_DPI = 200


## Notes: available variables and regions

Required model inputs: `LHFLX` (or `QFLX`, converted using latent heat of vaporization), `SHFLX`, `FSNS`, and `FLNS`. Cached derived variables: `Rn = FSNS - FLNS`, `LE`, `SH`, `Residual = Rn - (LE + SH)`, and `BowenRatio = SH / LE`.

Plot regions: `global`, `land`, and `ocean`. Existing member caches are reused unless `force_compute = True`.

Model input: analysis (`da`) runs restart every 6 h, so their analysis-time (`6hourly_da`) and `daily` values are first-model-step snapshots that do not represent surface fluxes; the budget uses the 6-h forecast means (`ts/6hourly`) aggregated to daily means (each day must have all 4 cycles). Forecast notebooks use daily files (continuous runs) and fall back to 6-hourly data.

Window: with `use_analysis_window = True`, analysis runs cover the 64-cycle days (S0: 16-31 Dec 2011; S1: 16-31 May 2012), matching the DA scorecard; the window is part of the cache file name.


## 3. Project setup


In [ ]:
import sys
import matplotlib.pyplot as plt
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.bias_metrics import experiment_registry
from util.atm_surface_energy_budget import INPUT_VARIABLES, OUTPUT_VARIABLES, REGIONS, energy_budget_cache_path, cache_surface_energy_budget, plot_energy_budget_timeseries
from util.da_scorecard import window_days


## 4. Read, process, and cache


In [ ]:
data_dir, figure_dir = workflow_output_dirs(
    workflow_name, "surface_energy_budget", output_root=DIAGNOSTIC_OUTPUT_ROOT
)
registry = experiment_registry(INPUT_DATA_ROOT, season, run_segment)
unknown = set(experiments) - set(registry)
if unknown:
    raise ValueError(
        f"Unavailable experiments: {sorted(unknown)}; choose from {list(registry)}"
    )
landmask = LAND_SUPPORT_DIR / "landmask_1x1.nc"
days = window_days(season) if (use_analysis_window and run_segment == "da") else None
window = None if days is None else (days[0], days[-1])
print("Window:", "whole month" if window is None else f"{window[0]:%Y-%m-%d} to {window[1]:%Y-%m-%d} ({len(days)} days)")

processed = {}
for experiment in experiments:
    meta = registry[experiment]
    count = meta["nens"] if max_members is None else min(max_members, meta["nens"])
    members = {}
    for number in range(1, count + 1):
        member = f"EN{number:02d}"
        target = energy_budget_cache_path(
            data_dir, experiment, member, meta["period"], window=window
        )
        existed = target.is_file()
        members[member] = cache_surface_energy_budget(
            target, data_root=INPUT_DATA_ROOT, experiment=experiment,
            season=season, run_segment=run_segment, member=member,
            period=meta["period"], window=window, force_compute=force_compute,
        )
        print("[REUSE]" if existed and not force_compute else "[WRITE]", target)
    processed[experiment] = members


## 5. Plot, save, and show


In [ ]:
figures = {}
for variable in plot_variables:
    fig = plot_energy_budget_timeseries(
        processed, variable, region, landmask,
        label_map={name: registry[name]["label"] for name in experiments},
    )
    output = figure_dir / f"{variable}_{region}_{season}_{run_segment}.png"
    fig.savefig(output, dpi=FIGURE_DPI, bbox_inches="tight")
    figures[variable] = fig
    if not show:
        plt.close(fig)
    print(f"[FIGURE] {output}")
